# CRDC Merged Knowledge Graph — Proof of Concept

This graph merges **three** NCI Cancer Research Data Commons (CRDC) portals into a
single queryable knowledge graph:

- **`gdc`** — Genomic Data Commons (molecular cases, samples, gene-level data)
- **`gc`**  — General Commons (pediatric/CCDI, HTAN, caNanoLab, imaging studies)
- **`pdc`** — Proteomics Data Commons (proteomic studies, deep case→sample→aliquot hierarchy)

Provenance is recorded on every node/edge in a multivalued `portal` list, so an entity
seen in more than one portal is tagged with *all* of them (e.g. `['pdc','gdc','gc']`).

**The point of this PoC:** show a capability that exists in the *merged* graph and in no
single source portal — traversing *across* portals through shared biological entities.
With PDC added, genes now converge across all three portals.

> ⚠️ **Load the right graph first.** The neo4j image imports from
> `graph_crdc_merged/graph/`. Make sure that holds the **3-portal merge** (the
> `merged_graph/` output of `python -m scripts.merge_graphs`), not a single-portal graph:
> ```bash
> cp merged_graph/nodes.tsv merged_graph/edges.tsv \
>    dglink/applications/mcp/graph_crdc_merged/graph/
> # rebuild + restart neo4j (from dglink/applications/)
> docker compose up -d --build neo-4j
> ```
> A quick sanity check: `MATCH (n) WHERE 'pdc' IN n.portal RETURN count(n)` should be > 0.

## 0. Connect

In [3]:
# pip install neo4j pandas   # if needed
import pandas as pd
from neo4j import GraphDatabase

URI = "bolt://localhost:7676"   # host-side; inside docker it is bolt://neo-4j:7687
AUTH = ("neo4j", "password")
PORTALS = ["gdc", "gc", "pdc"]

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

def q(cypher, **params):
    """Run a read query and return a tidy DataFrame."""
    records, _, _ = driver.execute_query(cypher, parameters_=params, database_="neo4j")
    return pd.DataFrame([r.data() for r in records])

# sanity check: all three portals present
q("""
UNWIND $portals AS p
MATCH (n) WHERE p IN n.portal
RETURN p AS portal, count(n) AS nodes ORDER BY nodes DESC
""", portals=PORTALS)

,portal,nodes
0,gdc,89905
1,pdc,36924
2,gc,5750


## 1. What's in the graph

Node types, and how many of each carry each portal tag. (A node can count under more
than one portal — that's the whole point.)

In [ ]:
q("""
MATCH (n)
UNWIND labels(n) AS label
RETURN label,
       count(*)                                       AS total,
       count(CASE WHEN 'gdc' IN n.portal THEN 1 END)  AS gdc,
       count(CASE WHEN 'gc'  IN n.portal THEN 1 END)  AS gc,
       count(CASE WHEN 'pdc' IN n.portal THEN 1 END)  AS pdc
ORDER BY total DESC
""")

,label,total,gdc,gc,pdc
0,biolink:Gene,32165,32050,2862,10649
1,biolink:MaterialSample,19136,54,0,19082
2,biolink:RNAProduct,7045,7044,11,9
3,biolink:Case,6327,3,134,6190
4,biolink:Disease,1623,4,1569,55
5,biolink:SmallMolecule,436,13,405,45
6,biolink:Study,332,2,64,266
7,biolink:Publication,310,0,310,0
8,biolink:NamedThing,189,2,160,32
9,biolink:MacromolecularComplex,102,27,80,10


## 2. Cross-portal overlap — the *mechanism*

The merge recognizes when an entity seen in different portals is the **same real-world
thing** and collapses it into one node whose `portal` list records every origin. These
multi-portal nodes are the bridges that make cross-portal questions possible.

Expected shape (from the `merged_graph` build): ~8,500 nodes shared by **pdc+gdc** and
**~2,100 shared by all three portals** — overwhelmingly genes.

In [ ]:
q("""
MATCH (n) WHERE n.portal IS NOT NULL
WITH n, [p IN $portals WHERE p IN n.portal] AS portals
WHERE size(portals) >= 2
RETURN apoc.text.join(portals, ' + ') AS shared_by,
       count(*)                       AS nodes
ORDER BY nodes DESC
""", portals=PORTALS)

,shared_by,nodes
0,gdc + pdc,8541
1,gdc + gc + pdc,2123
2,gdc + gc,651
3,gc + pdc,36


## 3. The headline: genes shared across **all three** portals

Adding PDC creates a genuine three-way convergence — genes independently present in the
genomic (GDC), general (GC) and proteomic (PDC) commons. That triangulation across data
modalities is something no single portal can produce.

In [ ]:
q("""
MATCH (g:`biolink:Gene`)
WHERE 'gdc' IN g.portal AND 'gc' IN g.portal AND 'pdc' IN g.portal
RETURN count(*) AS genes_in_all_three_portals
""")

,genes_in_all_three_portals
0,2116


In [ ]:
# a sample of the three-way genes
q("""
MATCH (g:`biolink:Gene`)
WHERE 'gdc' IN g.portal AND 'gc' IN g.portal AND 'pdc' IN g.portal
RETURN g.name AS gene, g.curie AS curie
ORDER BY g.name LIMIT 25
""")

,gene,curie
0,A2M,hgnc:7
1,AAGAB,hgnc:25662
2,AASDHPPT,hgnc:14235
3,ABCA1,hgnc:29
4,ABCC1,hgnc:51
5,ABCC3,hgnc:54
6,ABCC4,hgnc:55
7,ABCE1,hgnc:69
8,ABCF1,hgnc:70
9,ABCF3,hgnc:72


## 4. Gene footprint across portals

Pick a gene and see which projects/cases in each portal reference it. This is the
hypothesis-generation query: the same gene surfacing in a genomic cohort, a proteomic
study, and a general-commons study is a cross-modality signal worth a closer look.

> **Read these as co-occurrence / membership links** ("this gene is referenced in this
> project"), not as differential expression or a validated functional finding.

In [ ]:
q("""
MATCH (g:`biolink:Gene` {name:$gene})<-[:`biolink:related_to`]-(x)
WHERE x.portal IS NOT NULL
UNWIND [p IN $portals WHERE p IN x.portal] AS portal
RETURN portal, labels(x)[0] AS type, count(DISTINCT x) AS projects_or_cases
ORDER BY portal, projects_or_cases DESC
""", portals=PORTALS, gene="TP53")

,portal,type,projects_or_cases
0,gc,biolink:Study,1
1,gdc,biolink:Case,3
2,pdc,biolink:Study,2


## 5. Depth per node — case → sample provenance

Nodes aren't bare: cases carry their specimen lineage. PDC in particular contributes a
deep `Case → MaterialSample` hierarchy (thousands of samples/aliquots), so you can trace
from a case down to its physical specimens and their types.

In [ ]:
q("""
MATCH (c:`biolink:Case`)<-[:`biolink:derives_from`]-(m:`biolink:MaterialSample`)
WITH [p IN $portals WHERE p IN c.portal] AS portal, c, m
RETURN portal,
       count(DISTINCT c) AS cases,
       count(m)          AS samples,
       count(DISTINCT m.sample_type) AS distinct_sample_types
ORDER BY samples DESC
""", portals=PORTALS)

,portal,cases,samples,distinct_sample_types
0,[pdc],6190,9201,22
1,[gdc],3,54,4


## 6. Trace one concrete cross-portal path

Show the actual nodes a single three-way gene bridges. Paste the same pattern into
Neo4j Browser (`http://localhost:7474`) to get a rendered graph for a screenshot; set the
Gene caption to `name` so the shared node reads e.g. "TP53".

In [ ]:
q("""
MATCH (x)-[:`biolink:related_to`]->(g:`biolink:Gene` {name:$gene})
WHERE x.portal IS NOT NULL
WITH g, x, [p IN $portals WHERE p IN x.portal] AS xp
WHERE size(xp) > 0
RETURN xp[0]                       AS portal,
       labels(x)[0]                AS type,
       coalesce(x.name, x.curie)   AS project_or_case
ORDER BY portal LIMIT 30
""", portals=PORTALS, gene="TP53")

,portal,type,project_or_case
0,gc,biolink:Study,Human Tumor Atlas Network (HTAN) imaging data
1,gdc,biolink:Case,ncigdc:9b2c325c-1f03-43c7-ad5c-b49c6d205635
2,gdc,biolink:Case,ncigdc:0ee53efd-b992-4aeb-a091-8dd1bd32da6e
3,gdc,biolink:Case,ncigdc:0e9262d1-5aa8-4528-9aee-2815afcd23cd
4,pdc,biolink:Study,CPTAC UCEC Confirmatory Study - Acetylome
5,pdc,biolink:Study,CPTAC GBM Confirmatory Study - Acetylome


## 7. Provenance breakdown by portal membership

Extracted (DGLink file content) vs. exposed metadata, split by which portal(s) each
node belongs to — `gdc` only, `gc` only, `pdc + gdc`, all three, etc.

Two things to read off it:
- every **multi-portal** (shared) node is *extracted* content — exposed metadata stays
  portal-siloed, because a specific study/case/sample belongs to exactly one portal;
- **PDC-only** is almost entirely *metadata* (its deep specimen hierarchy), which is what
  rebalances the merged graph toward ~40% exposed metadata.

> Origin is inferred from `source`: `tabular_data`/`experimental_data` → extracted,
> everything else (`structural_information`, `clinical`, `metadata`) → exposed metadata.
> After the GDC `source` fix + regen, the previously-untagged GDC case/sample/disease
> nodes land in the `metadata` column instead of a blank origin.

In [ ]:
prov = q("""
MATCH (n) WHERE n.portal IS NOT NULL
WITH n,
     [p IN $portals WHERE p IN n.portal] AS combo,
     CASE WHEN any(s IN coalesce(n.source, []) WHERE s IN ['tabular_data','experimental_data'])
          THEN 'extracted' ELSE 'metadata' END AS origin
RETURN apoc.text.join(combo, ' + ') AS portal_membership, origin, count(*) AS nodes
""", portals=PORTALS)

pivot = prov.pivot_table(index='portal_membership', columns='origin',
                         values='nodes', aggfunc='sum', fill_value=0)
for col in ('extracted', 'metadata'):
    if col not in pivot.columns:
        pivot[col] = 0
pivot['total'] = pivot['extracted'] + pivot['metadata']
pivot = pivot.sort_values('total', ascending=False)
pivot

origin,extracted,metadata,total
portal_membership,,,
gdc,27832,62,27894
pdc,75,25577,25652
gdc + pdc,8541,0,8541
gc,1045,2063,3108
gdc + gc + pdc,2123,0,2123
gdc + gc,651,0,651
gc + pdc,36,0,36


## 8. Content breakdown: exposed metadata vs DGLink-extracted

The headline split — how much of the graph is content DGLink **extracted from files**
(`tabular_data` / `experimental_data`) vs. **project metadata the portals expose**
(`structural_information` / `clinical` / `metadata`) — with the same split broken out per
portal-membership combo (`gdc` only, `gdc + pdc`, all three, …).

The `— all portals —` row is the overall breakdown; `pct_metadata` is the share of each
row that is exposed metadata. Reads two ways: PDC-only is almost all metadata (its
specimen hierarchy), while every *shared* (multi-portal) combo is 100% extracted content.

In [ ]:
# self-contained: extracted vs exposed-metadata split, per portal-membership combo
prov = q("""
MATCH (n) WHERE n.portal IS NOT NULL
WITH n,
     [p IN $portals WHERE p IN n.portal] AS combo,
     CASE WHEN any(s IN coalesce(n.source, []) WHERE s IN ['tabular_data','experimental_data'])
          THEN 'extracted' ELSE 'metadata' END AS origin
RETURN apoc.text.join(combo, ' + ') AS portal_membership, origin, count(*) AS nodes
""", portals=PORTALS)

breakdown = prov.pivot_table(index='portal_membership', columns='origin',
                             values='nodes', aggfunc='sum', fill_value=0)
for col in ('extracted', 'metadata'):
    if col not in breakdown.columns:
        breakdown[col] = 0
breakdown['total'] = breakdown['extracted'] + breakdown['metadata']
breakdown = breakdown.sort_values('total', ascending=False)

# overall (all portals) as a total row, then the exposed-metadata share of each row
breakdown.loc['all portals'] = breakdown[['extracted', 'metadata', 'total']].sum()

breakdown['pct_metadata'] = (100 * breakdown['metadata'] / breakdown['total']).round(1)
breakdown

origin,extracted,metadata,total,pct_metadata
portal_membership,,,,
gdc,27832,62,27894,0.2
pdc,75,25577,25652,99.7
gdc + pdc,8541,0,8541,0.0
gc,1045,2063,3108,66.4
gdc + gc + pdc,2123,0,2123,0.0
gdc + gc,651,0,651,0.0
gc + pdc,36,0,36,0.0
all portals,40303,27702,68005,40.7


In [ ]:
driver.close()